# Parameter Tuning & Score Training

This notebook demonstrates how to:

1. Register programs and define parameters
2. Inject execution scores to simulate runs
3. Analyze per-parameter-set statistics
4. View and act on advisories
5. Compare parameter sets to find optimal configurations
6. Export trained knowledge for deployment

**All changes persist to the database** — scores, parameters, and
programs are immediately available in the GUI and API.

In [ ]:
from agent.notebook import Session

s = Session()
print(s)

## 1. Register a Program with Parameters

Register a face detection program with typed, constrained parameters.
Parameter format: `name:type:default:description`

In [ ]:
# Register the program (persists to DB)
prog = s.registry.register(
    name="facedetection",
    command_type="python",
    command_template="python facedetection.py --input {input_path} --threshold {threshold} --min-size {min_size}",
    description="Detects faces in video frames and filters clips by face count",
    purpose="Use after scene detection to filter clips containing people",
    required_inputs=["clip_path"],
    expected_outputs=["filtered_clips", "face_count"],
    tags=["filter", "face", "gpu"],
    parameters=[
        "threshold:float:0.5:Face detection confidence threshold (0.0-1.0)",
        "min_size:int:30:Minimum face size in pixels",
        "max_faces:int:100:Maximum faces to detect per frame",
    ],
)

# Pretty-print the registered program
s.registry.show("facedetection")

## 2. Register More Programs

Build up the registry with multiple programs.

In [ ]:
# Scene detection
s.registry.register(
    name="scene_detect",
    command_type="python",
    command_template="python scene_detect.py --input {video_path} --threshold {scene_threshold}",
    description="Detects scene cuts in a video",
    purpose="Split video into clips at scene boundaries",
    required_inputs=["video_path"],
    expected_outputs=["timecodes", "clips"],
    tags=["scene", "detection"],
    parameters=[
        "scene_threshold:float:0.3:Scene change sensitivity (lower = more sensitive)",
    ],
)

# FFmpeg resize
s.registry.register(
    name="ffmpeg_resize",
    command_type="ffmpeg",
    command_template="ffmpeg -i {input_path} -vf scale={width}:{height} -c:a copy {output_path}",
    description="Resize video to target resolution",
    purpose="Normalize video resolution before processing",
    required_inputs=["input_path"],
    expected_outputs=["resized_path"],
    tags=["ffmpeg", "scaling"],
    parameters=[
        "width:int:1280:Target width in pixels",
        "height:int:720:Target height in pixels",
    ],
)

# List all programs
for p in s.registry.list():
    print(f"  {p.name:20s} [{p.command_type.value}]  tags: {p.tags}")

## 3. Inject Training Scores

Simulate execution history by injecting scores for different
parameter combinations. This trains the advisory engine.

In [ ]:
# Good parameters: threshold=0.3 works well
s.scoring.inject_batch(
    "facedetection", "success",
    parameters={"threshold": 0.3, "min_size": 30},
    count=15,
    duration=2.5,
    output_size=500,
)

# Some failures at threshold=0.3 (but mostly good)
s.scoring.inject_batch(
    "facedetection", "failure",
    parameters={"threshold": 0.3, "min_size": 30},
    count=2,
    duration=0.5,
)

print("Injected 17 scores for threshold=0.3")

In [ ]:
# Bad parameters: threshold=0.9 produces zero output (too strict)
s.scoring.inject_batch(
    "facedetection", "zero_output",
    parameters={"threshold": 0.9, "min_size": 50},
    count=12,
    duration=3.0,
)

# A few successes at 0.9 (rare)
s.scoring.inject_batch(
    "facedetection", "success",
    parameters={"threshold": 0.9, "min_size": 50},
    count=2,
    duration=1.0,
)

print("Injected 14 scores for threshold=0.9")

In [ ]:
# Medium parameters: threshold=0.5 is OK but not great
s.scoring.inject_batch(
    "facedetection", "success",
    parameters={"threshold": 0.5, "min_size": 30},
    count=8,
    duration=2.0,
)
s.scoring.inject_batch(
    "facedetection", "zero_output",
    parameters={"threshold": 0.5, "min_size": 30},
    count=5,
    duration=2.0,
)

print("Injected 13 scores for threshold=0.5")

## 4. Analyze Results

View overall stats, per-parameter-set breakdowns, and advisories.

In [ ]:
# Full stats overview with parameter breakdown and advisories
s.scoring.show("facedetection")

In [ ]:
# Programmatic access to per-parameter-set stats
param_sets = s.scoring.param_sets("facedetection")

print("\nParameter set comparison:")
print(f"{'Parameters':<45} {'Success':>8} {'Runs':>5} {'Zero Out':>9}")
print("-" * 70)
for ps in sorted(param_sets, key=lambda x: x.success_rate, reverse=True):
    params_str = str(ps.parameters)
    if len(params_str) > 43:
        params_str = params_str[:40] + "..."
    print(
        f"{params_str:<45} {ps.success_rate:>7.0%} {ps.total_runs:>5} "
        f"{ps.zero_outputs:>9}"
    )

In [ ]:
# View advisories — the system recommends what to change
advisories = s.scoring.advisories("facedetection")

if advisories:
    print(f"\n{len(advisories)} advisory(ies) generated:\n")
    for a in advisories:
        print(f"  [{a.severity.value.upper()}] {a.title}")
        print(f"    {a.message}")
        if a.suggested_changes:
            print(f"    Suggested: {a.suggested_changes}")
        print()
else:
    print("No advisories (not enough data or no issues detected)")

## 5. Act on Advisories — Update Parameters

Based on the analysis, update the program's default parameter values.
These changes persist to the database.

In [ ]:
# The best-performing parameter set was threshold=0.3
# Set it as the current value (persisted to DB)
s.registry.set_param("facedetection", "threshold", 0.3)
s.registry.set_param("facedetection", "min_size", 30)

# Verify the change
prog = s.registry.get("facedetection")
for p in prog.parameters:
    val = p.current_value if p.current_value is not None else p.default
    print(f"  {p.name}: {val} (default: {p.default})")

In [ ]:
# Resolve the command with optimized parameters
prog = s.registry.get("facedetection")
resolved = prog.resolve_command()
print(f"Resolved command: {resolved}")

## 6. Iterate — Add More Scores and Re-evaluate

The training loop: adjust parameters, run more experiments, check advisories.

In [ ]:
# Clear old scores and start fresh with optimized parameters
deleted = s.scoring.clear("facedetection")
print(f"Cleared {deleted} old scores\n")

# Simulate runs with the optimized threshold=0.3
s.scoring.inject_batch("facedetection", "success", {"threshold": 0.3, "min_size": 30}, count=25, duration=2.5)
s.scoring.inject_batch("facedetection", "failure", {"threshold": 0.3, "min_size": 30}, count=1, duration=0.5)

# Check new stats
s.scoring.show("facedetection")

## 7. Add New Parameters to Existing Programs

Extend a program's parameter set without re-registering it.

In [ ]:
# Add a new parameter
s.registry.add_param("facedetection", "model:enum:yolov8:Detection model to use")

# Verify
s.registry.show("facedetection")

In [ ]:
# Remove a parameter that's no longer needed
s.registry.remove_param("facedetection", "model")

# Verify removal
prog = s.registry.get("facedetection")
print(f"Parameters: {[p.name for p in prog.parameters]}")

## 8. Export Everything

Export registry knowledge and scoring data for version control or deployment.

In [ ]:
# Export all programs (for git commit or sharing with team)
s.export.registry("configs/programs.json")

In [ ]:
# View what was exported
import json
from pathlib import Path

data = json.loads(Path("configs/programs.json").read_text())
for p in data:
    print(f"  {p['name']}: {len(p['parameters'])} params, tags={p['tags']}")

In [ ]:
s.close()